# Fine-tune SmolLM2-360M-Instruct on the Atelier curriculum

This notebook fine-tunes the small on-device chat model used by the Atelier site on your own course content (whyItMatters, coreIdea, mistakes, pro tips, quiz Q&A from all 4 curriculum files), so it answers course questions more specifically instead of generically.

**Before you start:** In Colab, go to `Runtime > Change runtime type` and pick a **T4 GPU** (free tier). This whole notebook takes roughly 15-30 minutes on a T4.


## 1. Install dependencies

In [ ]:
!pip install -q -U transformers accelerate peft trl datasets bitsandbytes optimum onnx onnxruntime


## 2. Load the training dataset

This was generated from your curriculum files (`whyItMatters`, `coreIdea`, `mistakes`, `proTips`, `checklist`, `nextStep`, and quiz Q&A) across all 4 curriculum files. It's hosted in your repo.

In [ ]:
import json, urllib.request

DATASET_URL = "https://raw.githubusercontent.com/aman-newbie/atelier-art-course/test/verify-workflow/training/atelier_qa_dataset.jsonl"
urllib.request.urlretrieve(DATASET_URL, "atelier_qa_dataset.jsonl")

examples = [json.loads(l) for l in open("atelier_qa_dataset.jsonl")]
print(f"Loaded {len(examples)} training examples")
print(json.dumps(examples[0], indent=2))


## 3. Load the base model

We fine-tune with LoRA (a lightweight adapter) instead of updating all 360M parameters directly \u2014 this needs far less GPU memory and trains much faster, while giving very similar quality for a dataset this size.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "HuggingFaceTB/SmolLM2-360M-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


## 4. Set up LoRA

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


## 5. Format the dataset with the model's chat template

In [ ]:
from datasets import Dataset

def format_example(ex):
    text = tokenizer.apply_chat_template(ex["messages"], tokenize=False, add_generation_prompt=False)
    return {"text": text}

ds = Dataset.from_list(examples).map(format_example)
print(ds[0]["text"][:500])


## 6. Train

In [ ]:
from trl import SFTTrainer, SFTConfig

sft_config = SFTConfig(
    output_dir="./atelier-smollm2-lora",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=20,
    save_strategy="epoch",
    bf16=True,
    dataset_text_field="text",
    max_seq_length=512,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=ds,
)

trainer.train()


## 7. Merge the LoRA adapter into the base model

transformers.js needs one plain model, not a base model + separate adapter.

In [ ]:
merged_model = model.merge_and_unload()
merged_model.save_pretrained("./atelier-smollm2-merged")
tokenizer.save_pretrained("./atelier-smollm2-merged")
print("Saved merged model.")


## 8. Quick sanity check before exporting

In [ ]:
from transformers import pipeline

test_pipe = pipeline("text-generation", model=merged_model, tokenizer=tokenizer, device=0)
messages = [
    {"role": "system", "content": "You are a friendly, encouraging drawing teacher's assistant embedded in the Atelier art course. The user is currently on the module \"Mindset & Introduction\"."},
    {"role": "user", "content": "What's a common mistake in this module?"},
]
out = test_pipe(messages, max_new_tokens=150, do_sample=False)
print(out[0]["generated_text"][-1]["content"])


## 9. Export to ONNX + quantize for transformers.js

This produces the exact file layout transformers.js expects:
```
onnx_out/
  config.json
  tokenizer.json
  tokenizer_config.json
  special_tokens_map.json
  onnx/
    model_quantized.onnx
```


In [ ]:
!optimum-cli export onnx --model ./atelier-smollm2-merged --task text-generation-with-past ./atelier_onnx_fp32


In [ ]:
!optimum-cli onnxruntime quantize --avx512 --onnx_model ./atelier_onnx_fp32 -o ./atelier_onnx_quantized


In [ ]:
import os, shutil

out_dir = "./atelier_onnx_final"
os.makedirs(os.path.join(out_dir, "onnx"), exist_ok=True)

# Config + tokenizer files at the root
for fname in ["config.json", "tokenizer.json", "tokenizer_config.json", "special_tokens_map.json", "generation_config.json"]:
    src = os.path.join("./atelier_onnx_fp32", fname)
    if os.path.exists(src):
        shutil.copy(src, out_dir)

# Find the quantized onnx file (name can vary slightly by optimum version) and
# rename it to model_quantized.onnx \u2014 transformers.js looks for this exact
# name when you request dtype: 'q8' in the site's pipeline() call.
for fname in os.listdir("./atelier_onnx_quantized"):
    if fname.endswith(".onnx"):
        shutil.copy(os.path.join("./atelier_onnx_quantized", fname), os.path.join(out_dir, "onnx", "model_quantized.onnx"))
        print("Copied", fname, "-> onnx/model_quantized.onnx")

print("\nFinal layout:")
for root, dirs, files in os.walk(out_dir):
    for f in files:
        print(os.path.join(root, f))


## 10. Get the model out of Colab

Easiest path (no Hugging Face token needed): zip the folder, download it, then create a **new model repo** on huggingface.co (Add model > paste your username/repo-name) and drag-and-drop the unzipped contents in through the website's "Files" tab.


In [ ]:
import shutil
shutil.make_archive("atelier_onnx_final", "zip", "atelier_onnx_final")

from google.colab import files
files.download("atelier_onnx_final.zip")


## 11. Point the site at your fine-tuned model

Once uploaded to `https://huggingface.co/YOUR_USERNAME/atelier-smollm2-finetuned`, change one line in `app.js`:

```js
const DOUBT_LOCAL_MODEL_ID = 'YOUR_USERNAME/atelier-smollm2-finetuned';
```

That's it \u2014 same pipeline code, just pointing at your fine-tuned weights instead of the stock model. Test locally before committing, since a bad merge/export step can produce a model that loads but generates garbage; the sanity check in step 8 is there to catch that *before* you export.
